In [ ]:
!git clone https://github.com/NikoGeor/actor-critic.git

In [ ]:
%cd actor-critic

In [ ]:
# Install dependencies
!pip install -r requirements.txt

# Install GPU-enabled JAX (Check your cluster's CUDA version first)
!pip install -U "jax[cuda12]"

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from mdptoolbox.example import forest
from mdptoolbox.mdp import ValueIteration

from src import MDP, ActorCriticAlgorithm

# Confirm JAX is utilizing the cluster's GPU
print("JAX Devices:", jax.devices())

In [ ]:
# 1. Define the Forest MDP
P, R = forest(S=3, r1=4, r2=2, p=0.1)
states = np.asarray([[0], [1], [2]])
actions = np.asarray([[0], [1]])
init_dist = np.asarray([1, 0, 0])
discount_factor = 0.7

mdp = MDP(states, actions, init_dist, R, P.transpose([1, 0, 2]), discount=discount_factor)

# 2. Compute the optimal policy baseline
vi = ValueIteration(
    transitions=P, reward=R, discount=discount_factor, 
    epsilon=1e-10, max_iter=100000, initial_value=0
)
vi.run()

optimal_policy = np.zeros((states.shape[0], actions.shape[0]))
for state_id in range(states.shape[0]):
    optimal_policy[state_id, vi.policy[state_id]] += 1

In [ ]:
# Hyperparameters
NEURONS = 5000
T = 100
ALPHA = 25
ZETA = 25
NUM_EXPERIMENTS = 30
betas = [0.500, 0.625, 0.750, 0.875, 1.000]

avg_rewards_per_beta = []
avg_actor_losses_per_beta = []

for beta in betas:
    print(f"\n--- Training for β = {beta:.3f} ---")
    rewards_for_this_beta = []
    actor_losses_for_this_beta = []
    
    for i in tqdm(range(NUM_EXPERIMENTS)):
        AC = ActorCriticAlgorithm(
            N=NEURONS, T=T, beta=beta, alpha=ALPHA, zeta=ZETA, mdp=mdp, seed=i
        )
        AC.fit(optimal_policy, mdp)
        
        rewards_for_this_beta.append(AC.rewards)
        actor_losses_for_this_beta.append(AC.actor_losses)
        
    # Average across the 30 independent runs (axis 0)
    avg_rewards_per_beta.append(np.mean(np.asarray(rewards_for_this_beta), axis=0))
    avg_actor_losses_per_beta.append(np.mean(np.asarray(actor_losses_for_this_beta), axis=0))

In [ ]:
plt.figure(figsize=(8, 6))
x = np.arange(0, 1, 0.01)
colors = ['red', 'orange', 'yellow', 'green', 'blue']

# Loop through the stored averages and plot them
for i, (beta, color) in enumerate(zip(betas, colors)):
    plt.plot(x, avg_rewards_per_beta[i], color=color, label=f'β = {beta:.3f}')

plt.xlabel('Training Progress')
plt.ylabel('Rewards')
plt.title('Expected Discounted Future Rewards')
plt.legend()
plt.grid(True, alpha=0.3)

# Display right in the notebook
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
x = np.arange(0, 1, 0.01)
colors = ['red', 'orange', 'yellow', 'green', 'blue']

for i, (beta, color) in enumerate(zip(betas, colors)):
    plt.plot(x, avg_actor_losses_per_beta[i], color=color, label=f'β = {beta:.3f}')

plt.xlabel('Training Progress')
plt.ylabel('Actor Loss')
plt.title('Actor MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()